In [ ]:
import datetime
print(datetime.datetime.now())

import sys
from pathlib import Path

current_path = Path.cwd()
root = current_path.parent
sys.path.append(str(root))

import tensorflow as tf

import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.set_visible_devices(gpus[0], 'GPU')
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
print(gpus)


In [ ]:
import keras

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

In [ ]:
warmup_steps = 1000
initial_lr = 1e-4
target_lr = 1e-6
decay_steps = 1000
batch_size = 512
num_epochs = 30
image_size = 28
channels = 1
num_classes = 100
input_shape = (28, 28)
patch_size = 7
num_patches = (image_size // patch_size) ** 2
projection_dim = 256

num_heads = 8
transformer_units = [
    projection_dim * 2,
    projection_dim,
]
transformer_layers = 6
mlp_head_units = [
    projection_dim * 8,
    projection_dim * 2,
]

class CustomSchedule(keras.optimizers.schedules.LearningRateSchedule):
  def __init__(self, d_model, warmup_steps=warmup_steps):
    super().__init__()

    self.d_model = d_model
    self.d_model = tf.cast(self.d_model, tf.float32)

    self.warmup_steps = warmup_steps

  def __call__(self, step):
    step = tf.cast(step, dtype=tf.float32)
    arg1 = tf.math.rsqrt(step)
    arg2 = step * (self.warmup_steps ** -1.5)

    return tf.math.rsqrt(self.d_model) * tf.math.minimum(arg1, arg2)
  

learning_rate = CustomSchedule(projection_dim * 2)

from keras.optimizers.schedules import CosineDecay


cosine_lr = CosineDecay(initial_lr, decay_steps,
                        alpha=1e-4,
                        warmup_target=target_lr,
                        warmup_steps = warmup_steps)

In [ ]:
import keras
from keras.layers import Dense, Dropout


def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = Dense(units, activation=keras.activations.gelu)(x)
        x = Dropout(dropout_rate)(x)
    return x

In [ ]:
import tensorflow as tf
import keras

class Patches(keras.layers.Layer):
    def __init__(self, patch_size):
        super().__init__()
        self.patch_size = patch_size

    def call(self, images):
        input_shape = images.shape
        batch_size = input_shape[0]
        height = input_shape[1]
        width = input_shape[2]
        channels = input_shape[3]
        num_patches_h = height // self.patch_size
        num_patches_w = width // self.patch_size


        patches = tf.image.extract_patches(images, 
                                           sizes=[1, self.patch_size, self.patch_size, 1], 
                                           strides=[1, self.patch_size, self.patch_size, 1], 
                                           rates=[1, 1, 1, 1], 
                                           padding='VALID')
        
        patches = tf.experimental.numpy.reshape(patches,
                             (
                                 -1,
                                 num_patches_h * num_patches_w,
                                 self.patch_size * self.patch_size * channels,
                             ), order='C')
        return patches
    
    def get_config(self):
        config = super().get_config()
        config.update({"patch_size":self.patch_size})
        return config

In [ ]:
from keras.layers import Embedding, Dense

class PatchEncoder(keras.layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super().__init__()
        self.num_patches = num_patches
        self.proejction = Dense(units=projection_dim)
        self.position_embedding = Embedding(input_dim=num_patches, output_dim=projection_dim)

    def call(self, patch):
        positions = tf.expand_dims(
            tf.experimental.numpy.arange(0, stop=self.num_patches, step=1), axis=0
        )
        projected_patches = self.proejction(patch)
        encoded = projected_patches + self.position_embedding(positions)
        return encoded
    
    def get_config(self):
        config = super().get_config()
        config.update({"num_patches":self.num_patches})
        return config

In [ ]:
from keras.layers import Reshape, Rescaling, Normalization, RandomFlip, RandomRotation, RandomZoom, LayerNormalization, Flatten, Dense, Dropout, MultiHeadAttention, Add, RandomErasing

data_augmentation = keras.Sequential(
    [
        Normalization(),
        Reshape((28, 28, 1), input_shape=(28, 28)),
        RandomFlip("horizontal"),
        RandomRotation(factor=0.02),
        RandomZoom(height_factor=0.2, width_factor=0.2),
        Rescaling(1./255, input_shape=(28, 28, 1))
    ],
    name="data_augmentation",
)

def create_vit_classifier():

    inputs = keras.Input(shape=input_shape)
    augmented = data_augmentation(inputs)
    patches = Patches(patch_size)(augmented)
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

    for _ in range(transformer_layers):
        x1 = LayerNormalization(epsilon=1e-6)(encoded_patches)
        attention_output = MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim, dropout=0.1)(x1, x1)
        x2 = Add()([attention_output, encoded_patches])
        x3 = LayerNormalization(epsilon=1e-6)(x2)
        x3 = mlp(x3, hidden_units=transformer_units, dropout_rate=0.1)
        encoded_patches = Add()([x3, x2])

    representation = LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = Flatten()(representation)
    representation = Dropout(0.5)(representation)

    features = mlp(representation, hidden_units=mlp_head_units, dropout_rate=0.5)
    logits = Dense(num_classes)(features)
    model = keras.Model(inputs=inputs, outputs=logits)
    return model

In [ ]:
def run_experiment(model, checkpoint_filepath):
    # optimizer = keras.optimizers.Adam(
    #         learning_rate=learning_rate,
    #         beta_1=0.9, 
    #         beta_2=0.999,
    #         epsilon=1e-9
    #     )
    optimizer = keras.optimizers.AdamW(
        learning_rate = learning_rate,
        weight_decay = 1e-3,
        beta_1 = 0.9,
        beta_2 = 0.999,
        epsilon = 1e-8,
    )

    model.compile(
        optimizer=optimizer,
        loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[
            keras.metrics.SparseCategoricalAccuracy(name="accuracy")
        ]
    )

    checkpoint_callback = keras.callbacks.ModelCheckpoint(
        checkpoint_filepath,
        monitor="val_accuracy",
        save_best_only=True,
        save_weights_only=True,
    )
    
    earlystop_callback=tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True
    )

    reducelr_callback=tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.5,
        patience=1,
        min_lr=1e-7
    )
    

    history = model.fit(
        x=x_train,
        y=y_train,
        batch_size=batch_size,
        epochs=num_epochs,
        validation_split=0.1,
        callbacks=[checkpoint_callback],
    )

    model.load_weights(checkpoint_filepath)
    _, accuracy = model.evaluate(x_test, y_test)
    print(f"Test accuracy:{round(accuracy * 100, 2)}%")

    return history

def plot_history(history, item):
    plt.plot(history.history[item], label=item)
    plt.plot(history.history["val_"+item], label="val_"+item)
    plt.xlabel("Epochs")
    plt.ylabel(item)
    plt.title("Train and Validation {} Over Epochs".format(item), fontsize=14)
    plt.legend()
    plt.grid()
    plt.show()

In [ ]:
vit_classifier = create_vit_classifier()
checkpoint_filepath="./tmp/mnist.mha6.p7.512.basline.weights.h5"
history = run_experiment(vit_classifier, checkpoint_filepath)
plot_history(history, "loss")
plot_history(history, "accuracy")

In [ ]:
np.save('./tmp/mnist.ViT.mha6.p7.512.baseline.history.npy', history.history)